In [ ]:
import os
from typing import Annotated, TypedDict, List
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate,  MessagesPlaceholder
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:

# --- 1. LLM CONFIGURATION ---
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Ministral-3-3B-Instruct-2512",
    huggingfacehub_api_token=os.getenv("HF_TOKEN"),
    temperature=0.1,
    max_new_tokens=512
)

In [ ]:
# --- 2. RETRIEVAL SETUP (ChromaDB + Semantic Search) ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_db = Chroma(
    collection_name="collection1",
    embedding_function=embeddings,
    host= os.environ.get("chroma_server", "localhost"),   # Chroma server address
    port = os.environ.get("chroma_port", 8000)   
)

# vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)    for local persistence
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

In [ ]:
# --- 3. STATE DEFINITION ---
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], "The chat history"]
    context: str
    status: str

In [ ]:
# --- 4. NODES / AGENT LOGIC ---

def retrieve_node(state: AgentState):
    
    """Fetches relevant documents based on user query."""

    last_message = state['messages'][-1].content
    docs = retriever.get_relevant_documents(last_message)
    context = "\n".join([doc.page_content for doc in docs])
    # Update status for streaming UI
    return {"context": context, "status": "Searching knowledge base..."}


from nemoguardrails import LLMRails, RailsConfig

# --- INITIALIZE GUARDRAILS ---
config = RailsConfig.from_path("./config")
# We pass the existing LLM instance to the rails engine

rails = LLMRails(config, llm=llm)

def call_model_node(state: AgentState):

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer consicely based ONLY on the provided context."
        "If the answer is not in the context, say you don't know.\n\n"
        "Context: \n{context}"
    ),
    MessagesPlaceholder(variable_name="history"), ("human", "{question}")
    ])
    
    # chain = prompt | llm
    
    # NeMo Guardrails would wrap this call or be a separate node
    response = rails.invoke({
        "context": state['context'],
        "history": state['messages'][:-1],
        "question": state['messages'][-1].content
    })
    
    return {
        "messages": [AIMessage(content=response)],
        "status": "Finalizing response..."
    }


'''Ver3
from nemoguardrails import LLMRails, RailsConfig
import asyncio

# --- 1. INITIALIZE GUARDRAILS ---
config = RailsConfig.from_path("./config")
rails = LLMRails(config, llm=llm)  # Pass existing LLM

# --- 2. RETRIEVE NODE ---
def retrieve_node(state: AgentState):
    """Fetch relevant documents based on the last user query."""
    last_message = state['messages'][-1].content
    docs = retriever.get_relevant_documents(last_message)
    context = "\n".join([doc.page_content for doc in docs])
    return {"context": context, "status": "Searching knowledge base..."}

# --- 3. CALL MODEL NODE WITH GUARDRAILS & PROMPT TEMPLATE ---
async def call_model_node(state: AgentState):
    """
    Uses LLM via NeMo Guardrails with ChatPromptTemplate and full conversation history.
    Preserves multi-turn context and enforces output safety/self-checks.
    """
    context = state['context']

    # Build system + multi-turn messages using your ChatPromptTemplate
    prompt_template = ChatPromptTemplate.from_messages([
        (
            "system",
            "Answer concisely based ONLY on the provided context. "
            "If the answer is not in the context, say you don't know.\n\n"
            "Context:\n{context}"
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}")
    ])

    # Prepare Guardrails message list
    messages = [{"role": "context", "content": context}]
    for msg in state['messages']:
        role = "user" if msg.type == "human" else "assistant"
        messages.append({"role": role, "content": msg.content})

    # Inject system prompt dynamically (simulate template formatting)
    # You could enhance by actually rendering the prompt template if needed
    messages.insert(1, {"role": "system", "content": prompt_template.format(context=context)})

    # Generate response with Guardrails
    res = await rails.generate_async(messages=messages)
    response_content = res.content

    # Status update for self-check / moderation
    status = "Finalizing response..."
    if not response_content or "I cannot answer" in response_content:
        status = "Response blocked by safety/fact-check guardrails."

    return {
        "messages": [AIMessage(content=response_content)],
        "status": status
    }

'''

In [ ]:
# --- 5. GRAPH ORCHESTRATION ---

workflow = StateGraph(AgentState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("llm", call_model_node)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "llm")
workflow.add_edge("llm", END)

# --- 6. MEMORY (PostgreSQL Persistence) ---

DB_USER = os.environ.get("PGSQL_USERNAME")
DB_PASSWORD = os.environ.get("PGSQL_PASSWORD")
DB_HOST = os.environ.get("PGSQL_HOST", "localhost")
DB_PORT = os.environ.get("PGSQL_PORT", "5432")
DB_NAME = os.environ.get("PGSQL_NAME")

DB_URI = (
    f"postgresql://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

checkpointer = PostgresSaver.from_conn_string(DB_URI)

# Compile the graph
app = workflow.compile(checkpointer=checkpointer)

# --- 7. API / FRONTEND CONNECTION (FastAPI) ---
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

api = FastAPI()

@api.post("/chat")
async def chat_endpoint(user_id: str, thread_id: str, message: str):
    config = {"configurable": {"thread_id": thread_id}}
    input_data = {"messages": [HumanMessage(content=message)]}
    
    async def event_generator():
        # Streams both status updates and final chunks to the React frontend
        async for event in app.astream(input_data, config=config, stream_mode="values"):
            if "status" in event:
                yield f"data: [STATUS] {event['status']}\n\n"
            if "messages" in event:
                yield f"data: {event['messages'][-1].content}\n\n"

    return StreamingResponse(event_generator(), media_type="text/event-stream")